# GoEmotions Feature Extraction

This notebook extracts only the affective GoEmotions sub-extractor output: `data/features/affective/goemotions_<split>.parquet`.

Expected input CSVs: `data/processed/train.csv`, `data/processed/val.csv`, and `data/processed/test.csv`.

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install transformers accelerate pandas pyarrow tqdm huggingface_hub

Mount Google Drive and point `PROJECT_DIR` to the project folder that contains `data/processed/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Change this path to your copied project folder in Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/mental_health_fusion')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
OUTPUT_DIR = PROJECT_DIR / 'data' / 'features' / 'affective'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COL = 'text'
POST_ID_COL = 'post_id'
BATCH_SIZE = 64
MODEL_NAME = 'SamLowe/roberta-base-go_emotions'

print('Project:', PROJECT_DIR)
print('Output:', OUTPUT_DIR)

If Hugging Face asks for authentication, create a read token on huggingface.co and run the login cell below.

In [ ]:
# Optional: uncomment if the model download needs authentication.
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import pipeline

GOEMOTIONS_LABELS = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral',
]

SENTENCE_RE = re.compile(r'(?<=[.!?])\s+')

def split_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return []
    return [s.strip() for s in SENTENCE_RE.split(text.strip()) if s.strip()]

def post_ids_for(df, split):
    if POST_ID_COL in df.columns:
        return df[POST_ID_COL].astype(str).tolist()
    return [f'{split}_{i}' for i in range(len(df))]

device = 0 if torch.cuda.is_available() else -1
emotion_pipe = pipeline(
    task='text-classification',
    model=MODEL_NAME,
    top_k=None,
    truncation=True,
    max_length=512,
    device=device,
)
print('Device:', 'cuda' if device == 0 else 'cpu')

In [ ]:
def extract_goemotions_texts(texts):
    outputs = []
    for text in tqdm(texts, desc='GoEmotions samples'):
        sentences = split_sentences(text)
        if not sentences:
            outputs.append(np.zeros(len(GOEMOTIONS_LABELS), dtype=np.float32))
            continue

        results = emotion_pipe(sentences, batch_size=BATCH_SIZE)
        rows = []
        for result in results:
            scores = {item['label']: float(item['score']) for item in result}
            rows.append([scores.get(label, 0.0) for label in GOEMOTIONS_LABELS])
        outputs.append(np.asarray(rows, dtype=np.float32).mean(axis=0))
    return outputs

def extract_split(split, force=False):
    input_path = PROCESSED_DIR / f'{split}.csv'
    output_path = OUTPUT_DIR / f'goemotions_{split}.parquet'
    if output_path.exists() and not force:
        print(f'Skipping existing: {output_path}')
        return output_path
    if not input_path.exists():
        raise FileNotFoundError(f'Missing input CSV: {input_path}')

    df = pd.read_csv(input_path)
    if TEXT_COL not in df.columns:
        raise KeyError(f'Missing text column {TEXT_COL!r} in {input_path}')

    features = extract_goemotions_texts(df[TEXT_COL].fillna('').astype(str).tolist())
    out = pd.DataFrame({'post_id': post_ids_for(df, split), 'features': features})
    out.to_parquet(output_path, index=False)
    print(f'Saved {len(out)} rows -> {output_path}')
    return output_path

In [ ]:
# Start with val to confirm the pipeline before running train/test.
extract_split('val', force=False)
extract_split('train', force=False)
extract_split('test', force=False)

In [ ]:
for split in ['train', 'val', 'test']:
    path = OUTPUT_DIR / f'goemotions_{split}.parquet'
    if path.exists():
        df = pd.read_parquet(path)
        dims = {len(x) for x in df['features']}
        print(split, len(df), dims)
        assert dims == {28}